# Inference Playground

Загрузка модели и генерация текста: `model.generate()`, chat template.

In [ ]:
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# Путь к сохранённой модели (pretrain/SFT из студии или ноутбука)
MODEL_DIR = Path('/app/out/notebook_llama_default')  # или /app/out/playground/module_playground_final
if not MODEL_DIR.exists():
    MODEL_DIR = Path('/app/models')  # скачанные HF модели
    print('Using', MODEL_DIR, '- put your model path in MODEL_DIR')
print('device:', DEVICE)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR if (MODEL_DIR / 'tokenizer_config.json').exists() else 'gpt2')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, trust_remote_code=True).to(DEVICE)
model.eval()
print('Model loaded')

In [ ]:
# Простая генерация
prompt = 'The meaning of life is'
inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
    )
text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Generated:', repr(text))

In [ ]:
# Chat template (если есть у токенизатора)
messages = [
    {"role": "user", "content": "What is 2+2?"},
]
if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7, pad_token_id=tokenizer.pad_token_id)
    reply = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print('Reply:', reply)
else:
    print('No chat_template; use plain prompt above.')